# Welcome to the Day 2 Lab!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Just before we get started --</h2>
            <span style="color:#f71;">I thought I'd take a second to point you at this page of useful resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

## First - let's talk about the Chat Completions API

1. The simplest way to call an LLM
2. It's called Chat Completions because it's saying: "here is a conversation, please predict what should come next"
3. The Chat Completions API was invented by OpenAI, but it's so popular that everybody uses it!

##### We will start by calling OpenAI/Gemini 

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

## [Gemini Guide](https://gemini.google.com/share/f2e7000468ba)

### What is an HTTP GET or POST?
- **HTTP** is the protocol that powers the web.
- **GET**: Retrieves data from a server. Think of visiting a website or fetching data. It’s like asking a librarian for a book. The parameters are usually visible in the URL (e.g., `?id=123`). Example: GET `https://api.example.com/users`. 

- **POST**: Used to send data to a server to create or update a resource. It’s like submitting a form. The data is sent in the "body" of the request, making it more secure and capable of handling complex data.

### What is an "Endpoint"?
An endpoint is a specific URL where an API can be accessed. Think of the API as a house and the endpoints as specific doors.

`https://api.example.com/v1/users` might be the door to get user data.

`https://api.example.com/v1/billing` might be the door for payment info.

#### The Rise of API Service Providers
In the past, developers had to build everything (databases, mail servers, AI models) from scratch. Today, Service Providers (like OpenAI, Google, or Stripe) "expose" their powerful infrastructure via endpoints. You don't need to own a supercomputer to use AI; you just need to know the right endpoint to send your data to.

APIs are now a core way that services expose functionality programmatically.
APIs let developers access data and services from platforms like:
- OpenAI (AI models)
- SendGrid (emails)
- Stripe (payments)
- Google Maps (locations)


---

## JSON (The Universal Language)
JSON (JavaScript Object Notation) is a lightweight format for storing and transporting data.

```
{
  "user": "student_01",
  "skills": ["Python", "SQL"],
  "active": true
}
```

### Why is it ubiquitous?

1. Language Independent: Almost every programming language can parse JSON into a dictionary or object.
2. Human Readable: It’s easy for developers to debug.
3. Lightweight: It has very little "syntax noise," making it faster to send over the internet than older formats like XML.

---

## API Keys (Your Digital Passport)
Because API providers spend money to run their servers, they need to know who is calling them. An API Key is a long string of random characters that acts as both a username and a password.

- Authentication: Proves you are who you say you are.
- Rate Limiting: Ensures you don't overwhelm the service (or helps them bill you if you go over your limit).
- Security: If a key is leaked, you can "revoke" it and generate a new one without changing your entire account password.



[Link to OpenAI to Gemini - Code conversion](https://gemini.google.com/share/9287154189ed)

In [ ]:
import requests
import os

# Get the key from the environment variable we just set
api_key = os.getenv("GOOGLE_API_KEY")

# Gemini uses a different URL structure than OpenAI
# Note the 'v1beta' and the model name in the URL
url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite:generateContent?key={api_key}"

headers = {
    "Content-Type": "application/json"
}

# The payload structure for Gemini is 'contents' -> 'parts' -> 'text'
payload = {
    "contents": [{
        "parts": [{"text": "Tell me a fun fact"}]
    }]
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)

# Display the result
response.json()

{'candidates': [{'content': {'parts': [{'text': "Sure, here's a fun fact for you:\n\n**A group of owls is called a parliament.**"}],
    'role': 'model'},
   'finishReason': 'STOP',
   'index': 0}],
 'usageMetadata': {'promptTokenCount': 5,
  'candidatesTokenCount': 22,
  'totalTokenCount': 27,
  'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 5}]},
 'modelVersion': 'gemini-2.5-flash-lite',
 'responseId': '-9PqaYroCZ3K4-EPss2f0Qo'}

Below is the code for openAI:

```
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()["choices"][0]["message"]["content"]
```


#### Key Differences to Watch Out For between Gemini and OpenAI

* **Authentication**: Unlike OpenAI, which puts the key in the `Authorization` header, Gemini often expects the key as a **query parameter** at the end of the URL (`?key=YOUR_KEY`).
* **Payload Schema**: Gemini doesn't use the `"messages": [{"role": "user", ...}]` format by default in this specific endpoint; it uses `"contents": [{"parts": [...]}]`.
* **Model Names**: In the code you shared, you had `gpt-5-nano` (a placeholder). For Gemini, common models are `gemini-1.5-flash` or `gemini-1.5-pro`.



OpenAI uses a **Header-based** approach, while Google’s Gemini API (via REST) primarily uses a **Resource-based URL** approach.

Here is the breakdown of why that URL looks so different from the OpenAI version you saw in Ed's notebook.

---

##### 1. The Anatomy of the Gemini URL
In the OpenAI code, the URL is a generic "endpoint" (`/v1/chat/completions`), and you tell the API which model you want inside the JSON payload. 

With Gemini, the model is actually part of the **web address** itself. Here is how that specific string is built:

* **`https://generativelanguage.googleapis.com/`**: This is the base host for Google’s Generative AI ecosystem.
* **`v1beta`**: This is the API version. Google often releases new features (like the latest Flash models) in the `v1beta` path before moving them to the stable `v1` path.
* **`models/gemini-1.5-flash`**: This is the **Resource Path**. You are telling the server exactly which "engine" to wake up before you even send your data.
* **`:generateContent`**: This is the **Method**. It tells the model what action to perform. (OpenAI implies this action by the folder name `/completions`).
* **`?key={api_key}`**: This is a **Query Parameter**. 


##### 2. Why the API Key is in the URL
This is the biggest security difference you'll notice:

* **OpenAI Style**: You pass the key in a "hidden" header (`Authorization: Bearer SK-...`). If you look at a raw network log, it’s tucked away in the metadata.
* **Gemini Style**: For simple REST requests, Google allows (and often expects) the key right in the URL string after the `?`. 


##### 3. Comparison Table

| Feature | OpenAI Structure | Gemini Structure |
| :--- | :--- | :--- |
| **Model Selection** | Inside the JSON `payload` | Inside the **URL Path** |
| **API Key Location** | `headers = {"Authorization": ...}` | `url = "...?key={api_key}"` |
| **Version Mapping** | Usually static (`/v1/`) | Changes based on stability (`/v1/` or `/v1beta/`) |
| **The "Action"** | Part of the path (`/chat/completions`) | Appended with a colon (`:generateContent`) |

---


Gemini's response is a bit more "nested" than OpenAI's, here is how you extract the actual text using Python in your notebook.

##### The Response Structure
When you call `response.json()`, Gemini sends back a dictionary. To get to the "Fun Fact," you have to dig through several layers.

##### How to extract the text





In [3]:
# Convert the raw response to a Python dictionary
data = response.json()

# Dig into the nesting: 
# candidates -> first item [0] -> content -> parts -> first item [0] -> text
fact = data['candidates'][0]['content']['parts'][0]['text']

print(f"Gemini says: {fact}")

Gemini says: Sure, here's a fun fact for you:

**A group of owls is called a parliament.**


##### Why is it so nested?
If you're wondering why Google makes you dig through so many layers compared to Ed's OpenAI examples, here’s the "why":

1.  **`candidates`**: Gemini can actually return multiple different versions of an answer at once (if you ask it to). We usually just take the first one at index `[0]`.
2.  **`content` & `parts`**: Gemini is "multimodal" from the ground up. A single response could technically contain text, code, and a reference to an image all in one go. Each of those pieces is a "part."
3.  **`text`**: This is the final layer where the actual string lives.

##### Pro-Tip for Windows/PowerShell Users
If you want to see the **entire** raw structure to explore it yourself, use this in your notebook cell:

```python
import json
print(json.dumps(data, indent=2))
```

This will print a beautifully formatted "tree" of the data. In Windows VS Code, you can even right-click that output and select **"Copy"** if you want to paste it into a file to study.


In [4]:
import json
print(json.dumps(data, indent=2))

{
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "text": "Sure, here's a fun fact for you:\n\n**A group of owls is called a parliament.**"
          }
        ],
        "role": "model"
      },
      "finishReason": "STOP",
      "index": 0
    }
  ],
  "usageMetadata": {
    "promptTokenCount": 5,
    "candidatesTokenCount": 22,
    "totalTokenCount": 27,
    "promptTokensDetails": [
      {
        "modality": "TEXT",
        "tokenCount": 5
      }
    ]
  },
  "modelVersion": "gemini-2.5-flash-lite",
  "responseId": "-9PqaYroCZ3K4-EPss2f0Qo"
}


Understanding the "plumbing" of a web request is essential. Even though the Python `requests` library makes it look like a simple function call, you are actually performing a standard web operation.

Here is the breakdown of those components from your Gemini code:

---

##### 1. The ENDPOINT
The **Endpoint** is the specific web address (URL) you are talking to. It’s like a phone number for a specific service.

* **In your code:**
    ```python
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?key={api_key}"
    ```
* **What it does:** It tells your computer exactly where the Gemini model is sitting on Google’s servers.

##### 2. The POST (The Method)
In web development, **GET** and **POST** are "verbs." 
* **GET** is like asking for a menu (you just receive data).
* **POST** is like placing an order (you send data—your prompt—and get a result back).

* **In your code:** ```python
    response = requests.post(url, headers=headers, json=payload)
    ```
* **Why POST?** Because you are sending a "payload" (your JSON prompt) to the server. You can't fit a large prompt inside a standard GET request easily.

##### 3. Where was the GET?
Actually, **there is no GET request in this specific snippet.** In AI engineering, we almost exclusively use **POST** because we need to send instructions to the model. 

However, if you wanted to see a **GET** request in action, you would use it to ask Google for a list of all available models:
```python
# A GET request example (just asking for info)
list_url = f"https://generativelanguage.googleapis.com/v1beta/models?key={api_key}"
models = requests.get(list_url) 
```

---

##### Summary Table for your Notebook

| Term | Location in Code | Analogy |
| :--- | :--- | :--- |
| **Endpoint** | The `url` string | The mailing address |
| **Method** | `requests.post()` | The "delivery type" (sending a package) |
| **Payload** | The `payload` dictionary | The letter inside the package |
| **Headers** | The `headers` dictionary | The instructions on the outside of the box |


### Why we separate the "instructions about the data" (headers) from the "actual data" (payload)?

This is a fundamental concept in networking that Ed touches on during the "under the hood" parts of the course. Think of it like the difference between **the envelope** and **the letter**.

The separation exists because the **Internet Infrastructure** and the **AI Model** need to know different things to do their jobs.


##### 1. The Headers (The Envelope)
The headers are for the **web servers and security layers** sitting between your Windows machine and Google’s data center. 

* **Routing**: Like an envelope's stamp and address, headers tell the server, "This is JSON data, send it to the JSON processing department."
* **Security**: It contains your "ID badge" (the API Key). The server checks this *before* it bothers to read your prompt. If your key is invalid, the server rejects the request immediately without wasting expensive GPU time on your prompt.
* **Metadata**: It tells the server how big the package is (`Content-Length`) and what language it speaks (`Content-Type`).

##### 2. The Payload (The Letter)
The payload is strictly for the **AI Model (Gemini/Gemma)** itself. 

* **The Instructions**: This is where your "Tell me a fun fact" lives. 
* **The Parameters**: This is where you tell the model how creative to be (Temperature, Top-P).
* **The Context**: This is where the history of your conversation lives.

##### Why not just put it all in one big block of text?

1. **Efficiency**: Web servers can read headers incredibly fast. They can decide to block, redirect, or cache your request just by looking at the first few bytes (the headers) without having to "open" and parse a giant 10,000-word prompt (the payload).
2. **Standardization**: By keeping headers separate, the same web server can handle a request from a Python script on Windows, a mobile app on an iPhone, or a website. They all use the same "envelope" format, even if the "letters" inside are totally different.
3. **Security**: Keeping the API Key in the headers (or URL) allows firewalls to inspect and validate your identity without having to look at your private data (the prompt).

---

##### Windows Technical Context
In your `uv` environment, when you use `requests.post(..., headers=headers, json=payload)`, the library is physically creating two separate sections of data in the "packet" it sends out over your Wi-Fi. 

If you ever get an error like **"415 Unsupported Media Type"**, it means the server read your **Header** ("envelope"), didn't like what it saw, and didn't even bother looking at your **Payload** ("letter").


##### A Quick Windows Tip for VS Code
If you want to see these requests happening in real-time, you can look at the **Network** tab if you were in a browser, but in a Python script, you are flying blind. 

To "see" the method and URL inside your code for debugging, you can always print the request object:
```python
print(f"Method used: {response.request.method}")
print(f"URL targeted: {response.request.url}")
```

### What is the openai/gemini package?

It's known as a Python Client Library. It's nothing more than a wrapper around making this exact call to the http endpoint. It just allows you to work with nice Python code instead of messing around with janky json objects.

But that's it. It's open-source and lightweight. Some people think it contains OpenAI model code - it doesn't!




The official corresponding package for Gemini is called **`google-genai`**.

In Ed's course, he uses the `openai` library. While Google used to have a library called `google-generativeai`, they have moved to the new **Google Gen AI SDK** (package name: `google-genai`) as the modern standard for 2026.

Here is how you set it up on your Windows machine to match Ed’s workflow.

---

### 1. Installation (The `uv` way)
In your PowerShell terminal inside VS Code, run:

```powershell
uv add google-genai
```

### 2. The Code Translation
Here is how you write the same "Fun Fact" code using the actual Gemini library instead of the raw `requests` method we discussed earlier:

```python
from google import genai
import os

# Initialize the client
# (It automatically looks for $env:GOOGLE_API_KEY on Windows)
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

# Make the call
response = client.models.generate_content(
    model="gemini-2.0-flash", 
    contents="Tell me a fun fact"
)

# Extracting text is much easier with the SDK!
print(response.text)
```

---

### Side-by-Side Comparison

| Feature | OpenAI (Ed's Code) | Gemini (Your Code) |
| :--- | :--- | :--- |
| **Package Name** | `openai` | `google-genai` |
| **Client Setup** | `OpenAI()` | `genai.Client()` |
| **Method Name** | `chat.completions.create()` | `models.generate_content()` |
| **Getting Text** | `res.choices[0].message.content` | `response.text` |



### Why use the package instead of `requests`?
The `requests` code I showed you earlier is great for learning the **Endpoints** and **Headers** (Week 1 stuff!), but the `google-genai` package is better for building real apps because:
1.  **Cleaner Code**: You don't have to manually build those long URLs.
2.  **Automatic Parsing**: You don't have to dig through `['candidates'][0]['content']` anymore; you just type `.text`.
3.  **Type Checking**: VS Code will give you better "Autofill" suggestions as you type.


In [5]:
from google import genai
import os

# Initialize the client
# (It automatically looks for $env:GOOGLE_API_KEY on Windows)
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

# Make the call
response = client.models.generate_content(
    model="gemini-2.5-flash-lite", 
    contents="Tell me a fun fact"
)

# Extracting text is much easier with the SDK!
print(response.text)

Here's a fun one:

**Honeybees can recognize human faces!**

Studies have shown that bees can learn to distinguish between different faces and even remember them for a period of time. They achieve this by analyzing patterns of light and dark, similar to how we process visual information. So, the next time you see a bee, it might just be giving you a knowing glance!


## And then this great thing happened:

OpenAI's Chat Completions API was so popular, that the other model providers created endpoints that are identical.

They are known as the "OpenAI Compatible Endpoints".

For example, google made one here: https://generativelanguage.googleapis.com/v1beta/openai/

And OpenAI decided to be kind: they said, hey, you can just use the same client library that we made for GPT. We'll allow you to specify a different endpoint URL and a different key, to use another provider.

So you can use:

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

And to be clear - even though OpenAI is in the code, we're only using this lightweight python client library to call the endpoint - there's no OpenAI model involved here.

If you're confused, please review Guide 9 in the Guides folder!

And now let's try it!

## THIS IS OPTIONAL - but if you wish to try out Google Gemini, please visit:

https://aistudio.google.com/

And set up your API key at

https://aistudio.google.com/api-keys

And then add your key to the `.env` file, being sure to Save the .env file after you change it:

`GOOGLE_API_KEY=AIz...`


In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")



In [ ]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

## And Ollama also gives an OpenAI compatible endpoint

...and it's on your local machine!

If the next cell doesn't print "Ollama is running" then please open a terminal and run `ollama serve`

In [ ]:
requests.get("http://localhost:11434").content

### Download llama3.2 from meta

Change this to llama3.2:1b if your computer is smaller.

Don't use llama3.3 or llama4! They are too big for your computer..

In [ ]:
!ollama pull llama3.2

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
# Get a fun fact

response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

In [ ]:
# Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

!ollama pull deepseek-r1:1.5b

In [ ]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
And in another Terminal (Mac) or Powershell (Windows), enter `ollama pull llama3.2`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

If Ollama is slow on your machine, try using `llama3.2:1b` as an alternative. Run `ollama pull llama3.2:1b` from a Terminal or Powershell, and change the code from `MODEL = "llama3.2"` to `MODEL = "llama3.2:1b"`